In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from PIL import Image
from tqdm import tqdm
import copy
import matplotlib.pyplot as plt

In [2]:
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection'

In [3]:
from Solar_Rooftop_Detection.accuracy import compute_metrics

In [4]:
class SegmentationDataset(Dataset):
    def __init__(self, root, image_folder="images", mask_folder="masks", transforms=None):
        self.root = root
        self.transforms = transforms
        self.image_folder = os.path.join(root, image_folder)
        self.mask_folder = os.path.join(root, mask_folder)
        self.image_names = sorted(os.listdir(self.image_folder))
        self.mask_names = sorted(os.listdir(self.mask_folder))

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_folder, self.image_names[idx])
        mask_path = os.path.join(self.mask_folder, self.mask_names[idx])
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Grayscale mask
        if self.transforms:
            image = self.transforms(image)
            mask = transforms.ToTensor()(mask)  # Mask to tensor (0-1 range)
        return {"image": image, "mask": mask}

In [5]:
def iou_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return intersection / (union + 1e-10) if union > 0 else 1.0

def f1_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    tp = np.sum(y_true * y_pred)
    fp = np.sum(y_pred) - tp
    fn = np.sum(y_true) - tp
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    return 2 * (precision * recall) / (precision + recall + 1e-10)

In [6]:
import pandas as pd

In [7]:
def evaluate_model(model, dataloader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Set to evaluation mode

    total_loss = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    criterion = torch.nn.BCEWithLogitsLoss()  # Same loss as training
    num_samples = 0
    all_matrics = []

    with torch.no_grad():  # No gradient computation
        for sample in tqdm(dataloader):
            inputs = sample["image"].to(device)
            masks = sample["mask"].to(device)
            outputs = model(inputs)["out"]  # Shape: (batch, 1, H, W)
            loss = criterion(outputs, masks)
            total_loss += loss.item() * inputs.size(0)

            # Convert logits to binary predictions
            
            preds = torch.sigmoid(outputs) > 0.5  # Threshold at 0.5
            preds_ = preds
            masks_ = masks
            preds = preds.cpu().numpy().astype(np.uint8)
            masks = masks.cpu().numpy().astype(np.uint8)

            # Compute metrics per batch
            for i in range(inputs.size(0)):
                all_matrics.append(compute_metrics(preds_[i], masks_[i]))
                # Compute IoU and F1 score
                total_iou += iou_score(masks[i], preds[i])
                total_f1 += f1_score(masks[i], preds[i])
            num_samples += inputs.size(0)

    avg_loss = total_loss / num_samples
    avg_iou = total_iou / num_samples
    avg_f1 = total_f1 / num_samples
    
    df = pd.DataFrame(all_matrics, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall","region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
    df.to_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_ROAD_SEGMENTATION/results/deeplabv3/metrics.csv", index=False)
    return avg_loss, avg_iou, avg_f1, all_matrics

In [8]:
data_dir = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_ROAD_SEGMENTATION/ROAD_DATASET/test" 
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
])

In [9]:
dataset = SegmentationDataset(data_dir, transforms=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

In [15]:
model = torch.load("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_ROAD_SEGMENTATION/results/deeplabv3/deeplab_rooftop_full_50.pth")
print("Model loaded successfully")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torchvision.models.segmentation.deeplabv3.DeepLabV3 was not an allowed global by default. Please use `torch.serialization.add_safe_globals([DeepLabV3])` or the `torch.serialization.safe_globals([DeepLabV3])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types a

In [16]:
avg_loss, avg_iou, avg_f1, matrixs_array = evaluate_model(model, dataloader)

NameError: name 'model' is not defined

In [ ]:
matrixs_array

In [ ]:
import pandas as pd
metrics_df = pd.read_csv("/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels/deeplabv3/metrics.csv")
metrics_df.head(5)
metrics_df.mean()

In [ ]:
avg_loss, avg_iou, avg_f1